In [1]:
# Enable interactive matplotlib rendering in Jupyter Notebook
%matplotlib widget

import matplotlib.pyplot as plt
from matplotlib.widgets import Slider
import numpy as np

import cv2
import sqlite3
from contextlib import contextmanager
from icecream import ic
from shapely import wkt
from shapely.geometry import Polygon
from shapely.wkt import loads



In [2]:
db_path= 'new.db'

In [3]:
@contextmanager
def open_db(db_path=db_path):
    conn = sqlite3.connect(db_path)
    try:
        # 1. Configure Row Factory (built-in dictionary-like rows)
        conn.row_factory = sqlite3.Row
        
        # 2. Load SpatiaLite Extension
        conn.enable_load_extension(True)
        conn.load_extension("mod_spatialite") 
        
        # Yield the fully configured connection
        yield conn
        
        conn.commit()
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()

# # Usage example:

# db_path = 'new.db'
# with open_db() as conn:
#     cursor = conn.cursor()
    
#     # You can now use SpatiaLite functions right away
#     cursor.execute("SELECT image_id FROM images WHERE damage_flag = 0")
    
#     # Because of sqlite3.Row, you can access columns by name!
#     image_id_queue = [row['image_id'] for row in cursor.fetchall()]    
    
#     # Connection is automatically committed and closed here.       

In [4]:
def draw_spatialite_contours(image, output_path):
    # 1. Load the background image
    # image = cv2.imread(image_path)
    # if image is None:
    #     raise FileNotFoundError(f"Could not load image from {image_path}")

    # # 2. Connect to the SpatiaLite database
    # conn = sqlite3.connect(db_path)
    # # Load the SpatiaLite extension depending on your OS
    # conn.enable_load_extension(True)
    # try:
    #     conn.load_extension("mod_spatialite")
    # except sqlite3.OperationalError:
    #     # Fallback for older configurations or specific environments
    #     conn.load_extension("spatialite")

    # cursor = conn.cursor()

    # # 3. Query the polygon geometries as WKT
    # # Replace 'your_table' and 'geom_column' with your actual table and column names
    # cursor.execute("SELECT AsText(geom_column) FROM your_table")
    # rows = cursor.fetchall()
    
    with open_db() as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT AsText(tree_poly) FROM trees WHERE image_id = 1")
        rows = cursor.fetchall()
    
    # 4. Loop through geometries and draw them
    for row in rows:
        wkt_string = row[0]
        if not wkt_string:
            continue

        # Parse WKT using Shapely
        polygon = wkt.loads(wkt_string)

        # Handle Polygon geometries
        if polygon.geom_type == "Polygon":
            polygons_to_draw = [polygon]
        elif polygon.geom_type == "MultiPolygon":
            polygons_to_draw = list(polygon.geoms)
        else:
            continue

        for poly in polygons_to_draw:
            # Extract exterior coordinates
            # Note: OpenCV expects integer coordinates (pixels)
            coords = np.array(poly.exterior.coords, dtype=np.int32)

            # Reshape coordinates to match OpenCV's required shape: (Number of vertices, 1, 2)
            pts = coords.reshape((-1, 1, 2))
            
            # ensure all coords are positive
            pts = np.abs(pts)
            ic(pts)
            

            # 5. Draw the contour line on the image
            # Parameters: image, points, isClosed, color (BGR format), thickness
            cv2.polylines(image, [pts], isClosed=True, color=(0, 255, 0), thickness=2)

            # Optional: If you want to fill the polygons instead, use fillPoly:
            # cv2.fillPoly(image, [pts], color=(0, 255, 0))

    # 6. Save and clean up
    cv2.imwrite(output_path, image)
    conn.close()
    print(f"Successfully drew contours and saved to {output_path}")


# Example usage:
# draw_spatialite_contours(img11, "output.png")


In [ ]:
def get_spatialite_contour(db_path, table_name, geometry_column, feature_id):
    """Downloads a WKT polygon from a Spatialite database and converts its
    exterior boundary into a cv2-compatible contour using Shapely.
    """
    # 1. Connect to Spatialite and fetch the geometry as WKT text
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Spatialite uses AsText() to convert binary geometry to WKT
    query = f"SELECT AsText({geometry_column}) FROM {table_name} WHERE rowid = ?"

    cursor.execute(query, (feature_id,))
    row = cursor.fetchone()
    conn.close()

    if not row or not row[0]:
        raise ValueError("No geometry found for the given ID.")

    wkt_string = row[0]

    # 2. Parse the WKT using Shapely
    geom = loads(wkt_string)

    if geom.geom_type not in ["Polygon", "MultiPolygon"]:
        raise ValueError(
            f"Unsupported geometry type: {geom.geom_type}. Expected Polygon or MultiPolygon."
        )

    # 3. Extract the exterior coordinates of the polygon
    # If it's a MultiPolygon, we'll take the first part for this single contour example
    if geom.geom_type == "MultiPolygon":
        # Extract the exterior of the first polygon in the collection
        exterior_coords = list(geom.geoms[0].exterior.coords)
    else:
        exterior_coords = list(geom.exterior.coords)

    # 4. Convert to NumPy integer array and reshape to OpenCV format: (N, 1, 2)
    # WKT repeats the starting point at the end. OpenCV doesn't need it,
    # so we slice it out using [:-1] to keep the contour clean.
    points = np.array(exterior_coords[:-1], dtype=np.int32)
    contour = points.reshape((-1, 1, 2))

    return contour


In [11]:
def save_cv2_contour_to_spatialite(db_path, table_name, geometry_column, contour, row_id=None):
    """Converts a cv2 contour into a WKT polygon and saves it to a Spatialite database.

    If a row_id is provided, it updates that specific row; otherwise, it inserts a new row.
    """
    # 1. Clean the cv2 contour shape from (N, 1, 2) to a flat list of coordinate tuples (N, 2)
    # OpenCV contours are deeply nested; .reshape(-1, 2) strips the extra 1-sized dimension.
    coords = contour.reshape((-1, 2))
    
    if len(coords) < 3:
        raise ValueError("A polygon requires at least 3 unique vertices.")

    # 2. Convert to a Shapely Polygon
    # Shapely automatically closes the geometric loop for the WKT export (repeating the 1st vertex at the end).
    poly = Polygon(coords)
    wkt_string = poly.wkt

    # 3. Connect to Spatialite and insert/update the geometry using GeomFromText
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    if row_id is not None:
        # Update existing row
        query = f"""
            UPDATE {table_name} 
            SET {geometry_column} = GeomFromText(?, 4326) 
            WHERE rowid = ?
        """
        cursor.execute(query, (wkt_string, row_id))
    else:
        # Insert a brand new row
        query = f"""
            INSERT INTO {table_name} ({geometry_column}) 
            VALUES (GeomFromText(?, 4326))
        """
        cursor.execute(query, (wkt_string,))

    conn.commit()
    conn.close()
    
    print(f"Successfully saved contour geometry to {table_name}.")


In [5]:
image_id = 2

# get image
with open_db() as conn:
    cursor = conn.cursor()
    cursor.execute(f'SELECT image_path FROM images WHERE image_id = {image_id} LIMIT 1')
    row = cursor.fetchone()
    image_path = row['image_path']   
img = cv2.imread(image_path)
cv2.imwrite('im1.png', img)

# add tree contours
with open_db() as conn:
    cursor = conn.cursor()
    cursor.execute(f'SELECT AsText(tree_poly) AS tree_poly_string FROM trees WHERE image_id = {image_id}')
    rows = cursor.fetchall()    
    
for row in rows:
    geom = loads(row['tree_poly_string']) # convert string to shapely object   
    exterior_coords = list(geom.exterior.coords)
    points = np.array(exterior_coords, dtype=np.int32)
    contour = points.reshape((-1, 1, 2))
    contour = np.abs(contour)
    contour = np.squeeze(contour)
    cv2.polylines(img, [contour], isClosed=True, color=(0, 255, 0), thickness=2)
cv2.imwrite('img2.png', img)

# draw damage contours
with open_db() as conn:
    cursor = conn.cursor()
    cursor.execute(f'SELECT AsText(damage_poly) AS damage_poly_string FROM damage WHERE image_id = {image_id}')
    rows = cursor.fetchall()
    
for row in rows:
    geom = loads(row['damage_poly_string']) # convert string to shapely object   
    exterior_coords = list(geom.exterior.coords)
    points = np.array(exterior_coords, dtype=np.int32)
    contour = points.reshape((-1, 1, 2))
    contour = np.abs(contour)
    contour = np.squeeze(contour)
    cv2.polylines(img, [contour], isClosed=True, color=(0, 0, 255), thickness=2)
cv2.imwrite('img3.png', img)
       
ic(contour)

ic| contour: array([[615, 360],
                    [616, 361],
                    [616, 362],
                    [617, 363],
                    [617, 364],
                    [617, 365],
                    [617, 366],
                    [617, 367],
                    [618, 368],
                    [618, 369],
                    [618, 370],
                    [618, 371],
                    [618, 372],
                    [618, 373],
                    [618, 374],
                    [619, 375],
                    [619, 376],
                    [619, 377],
                    [619, 378],
                    [619, 379],
                    [620, 380],
                    [620, 381],
                    [620, 382],
                    [620, 383],
                    [620, 384],
                    [620, 385],
                    [621, 386],
                    [621, 387],
                    [621, 388],
                    [621, 389],
                    [621, 390],
        

array([[615, 360],
       [616, 361],
       [616, 362],
       [617, 363],
       [617, 364],
       [617, 365],
       [617, 366],
       [617, 367],
       [618, 368],
       [618, 369],
       [618, 370],
       [618, 371],
       [618, 372],
       [618, 373],
       [618, 374],
       [619, 375],
       [619, 376],
       [619, 377],
       [619, 378],
       [619, 379],
       [620, 380],
       [620, 381],
       [620, 382],
       [620, 383],
       [620, 384],
       [620, 385],
       [621, 386],
       [621, 387],
       [621, 388],
       [621, 389],
       [621, 390],
       [622, 391],
       [623, 392],
       [624, 393],
       [625, 393],
       [626, 394],
       [627, 394],
       [628, 395],
       [629, 395],
       [630, 396],
       [631, 396],
       [632, 396],
       [633, 396],
       [634, 396],
       [635, 397],
       [636, 397],
       [637, 397],
       [638, 397],
       [639, 397],
       [640, 397],
       [640, 396],
       [639, 395],
       [639,

In [5]:
with open_db() as conn:
    cursor = conn.cursor()
    cursor.execute('SELECT image_path FROM images WHERE image_id = 1')
    row = cursor.fetchone()
    image_path = row['image_path']
    ic(image_path)

ic| image_path: '/home/aubrey/Desktop/crbdd/resources/example_images/08hs-palms-03-zglw-superJumbo.webp'


In [8]:
img11 = cv2.imread(image_path)
img11 = cv2.cvtColor(img11, cv2.COLOR_RGB2BGR)
# ic(img11);

draw_spatialite_contours(img11, "output.png")


Successfully drew contours and saved to output.png


ic| pts: array([[[486, 677]],
         
                [[486, 682]],
         
                [[485, 683]],
         
                ...,
         
                [[488, 678]],
         
                [[487, 678]],
         
                [[486, 677]]], shape=(1084, 1, 2), dtype=int32)
ic| pts: array([[[1035, 1273]],
         
                [[1035, 1274]],
         
                [[1034, 1275]],
         
                ...,
         
                [[1037, 1274]],
         
                [[1036, 1274]],
         
                [[1035, 1273]]], shape=(1066, 1, 2), dtype=int32)


In [7]:
# img12 - tree contours

with open_db() as conn:
    cursor = conn.cursor()
    cursor.execute('SELECT AsText(tree_poly) tree_wkt from trees WHERE image_id = 1')
    rows = cursor.fetchall()
    tree_wkts = [row['tree_wkt'] for row in rows]
    # ic(tree_wkts)
    
    img12 = np.zeros_like(img11)

    for tree_wkt in tree_wkts:
    
        # 1. Load the WKT into a Shapely geometry object
        polygon = wkt.loads(tree_wkt)

        # 2. Extract the exterior coordinates and convert to a NumPy array
        coords_array = np.array(polygon.exterior.coords)
        coords_array = np.abs(coords_array)  # Ensure all coordinates are positive
        # coords_array = np.squeeze(coords_array)
        ic(coords_array)
        
        im12 = cv2.drawContours(
            image=img12, 
            contours=coords_array, 
            contourIdx=-1,
            color=255, 
            thickness=3)
        


ic| coords_array: array([[486., 677.],
                         [486., 682.],
                         [485., 683.],
                         ...,
                         [488., 678.],
                         [487., 678.],
                         [486., 677.]], shape=(1084, 2))


error: OpenCV(4.13.0) /io/opencv/modules/imgproc/src/drawing.cpp:2564: error: (-215:Assertion failed) npoints > 0 in function 'drawContours'


In [ ]:
# Enable interactive matplotlib rendering in Jupyter Notebook
%matplotlib widget

import matplotlib.pyplot as plt
from matplotlib.widgets import Slider
import numpy as np

# 1. Create three synthetic images (or replace these with your loaded images)
# Image 1: Red background with a circle
X, Y = np.ogrid[:300, :300]
dist_from_center = np.sqrt((X - 150)**2 + (Y - 150)**2)
img1 = np.zeros((300, 300, 3), dtype=np.uint8)
img1[:, :, 0] = 200  # Red base
img1[dist_from_center < 100] = [255, 50, 50] # bright red circle

# Image 2: Green horizontal gradient
img2 = np.zeros((300, 300, 3), dtype=np.uint8)
gradient = np.linspace(0, 255, 300)
img2[:, :, 1] = gradient  # Green channel gradient

# Image 3: Blue grid pattern
img3 = np.zeros((300, 300, 3), dtype=np.uint8)
img3[::20, :, 2] = 255  # Blue grid lines
img3[:, ::20, 2] = 255

img1 = img11
img2 = img11
img3 = img11


# 2. Set up the Matplotlib figure and axis layout
fig, ax = plt.subplots(figsize=(7, 7))
plt.subplots_adjust(bottom=0.25)  # Leave space at the bottom for sliders

# Set initial transparency values
alpha1_init, alpha2_init, alpha3_init = 0.8, 0.5, 0.3

# Display the images on the same axis (overlaid in order: 1 -> 2 -> 3)
im1 = ax.imshow(img1, alpha=alpha1_init)
im2 = ax.imshow(img2, alpha=alpha2_init)
im3 = ax.imshow(img3, alpha=alpha3_init)
ax.axis('off')  # Hide axis ticks

# 3. Create Slider axes positions [left, bottom, width, height]
ax_alpha1 = plt.axes([0.2, 0.15, 0.65, 0.03])
ax_alpha2 = plt.axes([0.2, 0.10, 0.65, 0.03])
ax_alpha3 = plt.axes([0.2, 0.05, 0.65, 0.03])

# Create the Sliders
slider1 = Slider(ax_alpha1, 'Layer 1 (Base)', 0.0, 1.0, valinit=alpha1_init)
slider2 = Slider(ax_alpha2, 'Layer 2 (Mid)',  0.0, 1.0, valinit=alpha2_init)
slider3 = Slider(ax_alpha3, 'Layer 3 (Top)',  0.0, 1.0, valinit=alpha3_init)

# 4. Define update callback function
def update(val):
    im1.set_alpha(slider1.val)
    im2.set_alpha(slider2.val)
    im3.set_alpha(slider3.val)
    fig.canvas.draw_idle()

# Attach sliders to update function
slider1.on_changed(update)
slider2.on_changed(update)
slider3.on_changed(update)

plt.show()